In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import Checkbox, RadioButtons, IntSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# COMPARISON OF IDEAL ANALOG FILTER APPROXIMATIONS
#
# Interactive parameter:
#
#       N = filter order, 1,...,10
#
# Fixed reference frequency:
#
#       ω = 1 rad/s
#
# Additional fixed parameters:
#
#       Chebyshev I : Ap = 1 dB
#       Chebyshev II: As = 40 dB
#       Elliptic    : Ap = 1 dB, As = 40 dB
#
# Filters:
#
#       Butterworth
#       Chebyshev I
#       Chebyshev II
#       Elliptic / Cauer
#       Bessel-Thomson
#
# Quantities:
#
#       Magnitude Response
#       Phase Response
#       Gain Function
#       Loss Function
#       Group Delay
#       Impulse Response
#       Step Response
#
# Every change of filter order reconstructs all five analog filters and
# recalculates all seven responses.
#
# If no filter is selected:
#
#       - the plot displays an instructional message
#       - the response selector is disabled
#
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}

</style>
"""))

# ==============================================================================
# FIXED PARAMETERS
# ==============================================================================

wc = 1.0
Ap = 1.0
As = 40.0

# ==============================================================================
# FREQUENCY AND TIME AXES
# ==============================================================================

omega = np.logspace(-2, 1.2, 4000)

t = np.linspace(0.0, 20.0, 4000)

# ==============================================================================
# FILTER NAMES AND COLORS
# ==============================================================================

filter_names = ['Butterworth', 'Chebyshev I', 'Chebyshev II', 'Elliptic / Cauer', 'Bessel-Thomson']

filter_colors = {
    'Butterworth': 'red',
    'Chebyshev I': 'blue',
    'Chebyshev II': 'green',
    'Elliptic / Cauer': 'purple',
    'Bessel-Thomson': 'orange'
}

# ==============================================================================
# RESPONSE STORAGE
# ==============================================================================

responses = {}

# ==============================================================================
# FILTER CONSTRUCTION AND RESPONSE CALCULATION
# ==============================================================================

def calculate_all_responses(N):

    new_responses = {}

    filters = {}

    filters['Butterworth'] = signal.butter(N, wc, btype='low', analog=True, output='ba')

    filters['Chebyshev I'] = signal.cheby1(N, Ap, wc, btype='low', analog=True, output='ba')

    filters['Chebyshev II'] = signal.cheby2(N, As, wc, btype='low', analog=True, output='ba')

    filters['Elliptic / Cauer'] = signal.ellip(N, Ap, As, wc, btype='low', analog=True, output='ba')

    filters['Bessel-Thomson'] = signal.bessel(N, wc, btype='low', analog=True, output='ba', norm='mag')

    for name, (b, a) in filters.items():

        _, H = signal.freqs(b, a, worN=omega)

        magnitude = np.abs(H)

        phase = np.unwrap(np.angle(H))

        phase_deg = np.rad2deg(phase)

        gain_db = 20.0 * np.log10(np.maximum(magnitude, 1e-15))

        loss_db = -gain_db

        group_delay = -np.gradient(phase, omega)

        system = signal.TransferFunction(b, a)

        t_impulse, impulse = signal.impulse(system, T=t)

        t_step, step = signal.step(system, T=t)

        new_responses[name] = {
            'Magnitude Response': (omega, magnitude),
            'Phase Response': (omega, phase_deg),
            'Gain Function': (omega, gain_db),
            'Loss Function': (omega, loss_db),
            'Group Delay': (omega, group_delay),
            'Impulse Response': (t_impulse, impulse),
            'Step Response': (t_step, step)
        }

    return new_responses

# ==============================================================================
# INITIAL FILTER ORDER
# ==============================================================================

N_initial = 4

responses = calculate_all_responses(N_initial)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML(f"""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1320px;
    max-width:1320px;
    box-sizing:border-box;
">
<b>Comparison of Ideal Analog Filter Approximations</b><br>
The filter order N can be varied from 1 to 10.
All filters use a normalized reference frequency of
<b>ω = {wc:.1f} rad/s</b>.
For Chebyshev I and elliptic filters,
A<sub>p</sub> = {Ap:.1f} dB.
For Chebyshev II and elliptic filters,
A<sub>s</sub> = {As:.0f} dB.
<br>
<b>Interpretation:</b>
Use the checkboxes to select the filters, the radio buttons to select the
quantity to be compared, and the horizontal slider to investigate the effect
of filter order. The same filter always retains the same color.
</div>
""", layout=Layout(width='1330px', max_width='1330px'))

# ==============================================================================
# CHECKBOXES
# ==============================================================================

cb_butter = Checkbox(value=True, description='Butterworth', indent=False, layout=Layout(width='145px'))

cb_cheby1 = Checkbox(value=True, description='Chebyshev I', indent=False, layout=Layout(width='145px'))

cb_cheby2 = Checkbox(value=True, description='Chebyshev II', indent=False, layout=Layout(width='145px'))

cb_ellip = Checkbox(value=True, description='Elliptic / Cauer', indent=False, layout=Layout(width='145px'))

cb_bessel = Checkbox(value=True, description='Bessel-Thomson', indent=False, layout=Layout(width='145px'))

checkboxes = {
    'Butterworth': cb_butter,
    'Chebyshev I': cb_cheby1,
    'Chebyshev II': cb_cheby2,
    'Elliptic / Cauer': cb_ellip,
    'Bessel-Thomson': cb_bessel
}

# ==============================================================================
# COLOR INDICATORS
# ==============================================================================

def color_indicator(color):

    return HTML(f"""
    <div style="
        width:42px;
        height:12px;
        display:flex;
        align-items:center;
        margin-left:4px;
    ">
        <div style="
            width:34px;
            height:3px;
            background:{color};
            border-radius:2px;
        "></div>
    </div>
    """, layout=Layout(width='50px'))

row_butter = HBox([cb_butter, color_indicator(filter_colors['Butterworth'])], layout=Layout(align_items='center'))

row_cheby1 = HBox([cb_cheby1, color_indicator(filter_colors['Chebyshev I'])], layout=Layout(align_items='center'))

row_cheby2 = HBox([cb_cheby2, color_indicator(filter_colors['Chebyshev II'])], layout=Layout(align_items='center'))

row_ellip = HBox([cb_ellip, color_indicator(filter_colors['Elliptic / Cauer'])], layout=Layout(align_items='center'))

row_bessel = HBox([cb_bessel, color_indicator(filter_colors['Bessel-Thomson'])], layout=Layout(align_items='center'))

# ==============================================================================
# PARAMETER INFORMATION
# ==============================================================================

parameter_html = HTML()

def update_parameter_html(N):

    parameter_html.value = f"""
    <div style="
        margin-top:8px;
        padding-top:8px;
        border-top:1px solid #dddddd;
        font-size:11px;
        line-height:1.65;
    ">
    <b>Current parameters</b><br>
    N = <span style="color:#0066cc;"><b>{N}</b></span><br>
    ω<sub>c</sub> = {wc:.1f} rad/s<br>
    A<sub>p</sub> = {Ap:.1f} dB<br>
    A<sub>s</sub> = {As:.0f} dB
    </div>
    """

update_parameter_html(N_initial)

# ==============================================================================
# FILTER-SELECTION PANEL
# ==============================================================================

filter_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:7px;
">
Filter Selection
</div>
""")

filter_panel = VBox(
    [
        filter_title,
        row_butter,
        row_cheby1,
        row_cheby2,
        row_ellip,
        row_bessel,
        parameter_html
    ],
    layout=Layout(
        width='240px',
        min_width='240px',
        max_width='240px',
        border='1px solid #cccccc',
        padding='10px',
        align_items='flex-start'
    )
)

# ==============================================================================
# RESPONSE SELECTOR
# ==============================================================================

response_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:8px;
">
Displayed Quantity
</div>
""")

response_selector = RadioButtons(
    options=[
        'Magnitude Response',
        'Phase Response',
        'Gain Function',
        'Loss Function',
        'Group Delay',
        'Impulse Response',
        'Step Response'
    ],
    value='Magnitude Response',
    description='',
    layout=Layout(width='190px')
)

# ==============================================================================
# RESPONSE PANEL
#
# A small left margin is added so that the radio-button frame is visually
# separated from the plot. The vertical position remains unchanged.
# ==============================================================================

response_panel = VBox(
    [
        response_title,
        response_selector
    ],
    layout=Layout(
        width='210px',
        min_width='210px',
        max_width='210px',
        border='1px solid #cccccc',
        padding='10px',
        margin='0px 0px 0px 18px',
        align_items='flex-start'
    )
)

# ==============================================================================
# ORDER SLIDER
# ==============================================================================

order_slider = IntSlider(
    value=N_initial,
    min=1,
    max=10,
    step=1,
    description='Filter Order N:',
    continuous_update=True,
    readout=True,
    style={'description_width':'95px'},
    layout=Layout(width='650px')
)

order_slider_box = HBox(
    [order_slider],
    layout=Layout(
        width='820px',
        padding='0px 0px 0px 72px',
        box_sizing='border-box',
        justify_content='flex-start',
        align_items='center',
        margin='-8px 0px 0px 0px'
    )
)

# ==============================================================================
# MAIN FIGURE
# ==============================================================================

fig, ax = plt.subplots(figsize=(8.4, 5.4))

lines = {}

for name in filter_names:

    lines[name], = ax.plot([], [], linewidth=2.2, color=filter_colors[name], label=name)

reference_line = ax.axvline(1.0, color='black', linestyle=':', linewidth=1.0)

zero_line = ax.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

# ==============================================================================
# MESSAGE BOX FOR EMPTY SELECTION
# ==============================================================================

empty_message = ax.text(
    0.5,
    0.5,
    'Select at least one filter type.',
    transform=ax.transAxes,
    horizontalalignment='center',
    verticalalignment='center',
    fontsize=13,
    fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.7', facecolor='white', edgecolor='gray', alpha=0.95)
)

empty_message.set_visible(False)

# ==============================================================================
# FIGURE SETTINGS
# ==============================================================================

ax.grid(True, which='both', linestyle=':', alpha=0.5)

ax.tick_params(axis='both', labelsize=9)

fig.subplots_adjust(left=0.12, right=0.97, bottom=0.18, top=0.88)

fig.canvas.header_visible = False

fig.canvas.toolbar_visible = False

fig.canvas.resizable = False

fig.canvas.layout.width = '820px'

fig.canvas.layout.height = '530px'

# ==============================================================================
# PLOT CONFIGURATION
# ==============================================================================

plot_settings = {

    'Magnitude Response': {
        'title': 'Comparison of Magnitude Responses',
        'xlabel': 'Angular Frequency ω (rad/s)',
        'ylabel': '|H(jω)|',
        'xscale': 'log',
        'xlim': (0.01, 15.0),
        'ylim': (0.0, 1.08),
        'reference': True,
        'zero': False
    },

    'Phase Response': {
        'title': 'Comparison of Phase Responses',
        'xlabel': 'Angular Frequency ω (rad/s)',
        'ylabel': 'Phase (degrees)',
        'xscale': 'log',
        'xlim': (0.01, 15.0),
        'ylim': (-910.0, 40.0),
        'reference': True,
        'zero': True
    },

    'Gain Function': {
        'title': 'Comparison of Gain Functions',
        'xlabel': 'Angular Frequency ω (rad/s)',
        'ylabel': 'Gain G(ω) (dB)',
        'xscale': 'log',
        'xlim': (0.01, 15.0),
        'ylim': (-200.0, 5.0),
        'reference': True,
        'zero': True
    },

    'Loss Function': {
        'title': 'Comparison of Loss Functions',
        'xlabel': 'Angular Frequency ω (rad/s)',
        'ylabel': 'Loss A(ω) (dB)',
        'xscale': 'log',
        'xlim': (0.01, 15.0),
        'ylim': (0.0, 200.0),
        'reference': True,
        'zero': True
    },

    'Group Delay': {
        'title': 'Comparison of Group Delays',
        'xlabel': 'Angular Frequency ω (rad/s)',
        'ylabel': 'Group Delay τ(ω)',
        'xscale': 'log',
        'xlim': (0.01, 15.0),
        'ylim': (-5.0, 30.0),
        'reference': True,
        'zero': True
    },

    'Impulse Response': {
        'title': 'Comparison of Impulse Responses',
        'xlabel': 'Time t (s)',
        'ylabel': 'h(t)',
        'xscale': 'linear',
        'xlim': (0.0, 20.0),
        'ylim': (-1.0, 1.5),
        'reference': False,
        'zero': True
    },

    'Step Response': {
        'title': 'Comparison of Step Responses',
        'xlabel': 'Time t (s)',
        'ylabel': 'Amplitude',
        'xscale': 'linear',
        'xlim': (0.0, 20.0),
        'ylim': (-0.1, 1.7),
        'reference': False,
        'zero': False
    }
}

# ==============================================================================
# UPDATE DISPLAYED PLOT
# ==============================================================================

def update_plot(change=None):

    selected_filters = [name for name in filter_names if checkboxes[name].value]

    # --------------------------------------------------------------------------
    # NO FILTER SELECTED
    # --------------------------------------------------------------------------

    if len(selected_filters) == 0:

        for name in filter_names:

            lines[name].set_data([], [])

        empty_message.set_visible(True)

        response_selector.disabled = True

        reference_line.set_visible(False)

        zero_line.set_visible(False)

        ax.set_title('Filter Comparison', fontsize=13, fontweight='bold', pad=8)

        ax.set_xlabel('')

        ax.set_ylabel('')

        ax.set_xscale('linear')

        ax.set_xlim(0.0, 1.0)

        ax.set_ylim(0.0, 1.0)

        ax.grid(False)

        fig.canvas.draw_idle()

        return

    # --------------------------------------------------------------------------
    # AT LEAST ONE FILTER SELECTED
    # --------------------------------------------------------------------------

    empty_message.set_visible(False)

    response_selector.disabled = False

    selected_response = response_selector.value

    settings = plot_settings[selected_response]

    # --------------------------------------------------------------------------
    # UPDATE CURVES
    # --------------------------------------------------------------------------

    for name in filter_names:

        if checkboxes[name].value:

            x_values, y_values = responses[name][selected_response]

            y_values = np.array(y_values, copy=True)

            if selected_response == 'Group Delay':

                y_values[np.abs(y_values) > 100.0] = np.nan

            lines[name].set_data(x_values, y_values)

        else:

            lines[name].set_data([], [])

    # --------------------------------------------------------------------------
    # AXIS SCALE
    # --------------------------------------------------------------------------

    ax.set_xscale(settings['xscale'])

    ax.set_xlim(settings['xlim'])

    # --------------------------------------------------------------------------
    # DYNAMIC PHASE RANGE
    # --------------------------------------------------------------------------

    if selected_response == 'Phase Response':

        current_N = order_slider.value

        phase_min = -90.0 * current_N

        ax.set_ylim(1.02 * phase_min, 20.0)

    else:

        ax.set_ylim(settings['ylim'])

    # --------------------------------------------------------------------------
    # LABELS
    # --------------------------------------------------------------------------

    ax.set_title(settings['title'], fontsize=13, fontweight='bold', pad=8)

    ax.set_xlabel(settings['xlabel'], fontsize=10)

    ax.set_ylabel(settings['ylabel'], fontsize=10)

    # --------------------------------------------------------------------------
    # REFERENCE LINES
    # --------------------------------------------------------------------------

    reference_line.set_visible(settings['reference'])

    zero_line.set_visible(settings['zero'])

    if selected_response == 'Step Response':

        zero_line.set_visible(True)

        zero_line.set_ydata([1.0, 1.0])

    else:

        zero_line.set_ydata([0.0, 0.0])

    # --------------------------------------------------------------------------
    # GRID
    # --------------------------------------------------------------------------

    ax.grid(True, which='both', linestyle=':', alpha=0.5)

    # --------------------------------------------------------------------------
    # REDRAW
    # --------------------------------------------------------------------------

    fig.canvas.draw_idle()

# ==============================================================================
# UPDATE FILTER ORDER
# ==============================================================================

def update_order(change=None):

    global responses

    current_N = order_slider.value

    # --------------------------------------------------------------------------
    # RECONSTRUCT ALL FILTERS AND RESPONSES
    # --------------------------------------------------------------------------

    responses = calculate_all_responses(current_N)

    # --------------------------------------------------------------------------
    # UPDATE PARAMETER PANEL
    # --------------------------------------------------------------------------

    update_parameter_html(current_N)

    # --------------------------------------------------------------------------
    # UPDATE CURRENT PLOT
    # --------------------------------------------------------------------------

    update_plot()

# ==============================================================================
# CALLBACKS
# ==============================================================================

cb_butter.observe(update_plot, names='value')

cb_cheby1.observe(update_plot, names='value')

cb_cheby2.observe(update_plot, names='value')

cb_ellip.observe(update_plot, names='value')

cb_bessel.observe(update_plot, names='value')

response_selector.observe(update_plot, names='value')

order_slider.observe(update_order, names='value')

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_plot()

# ==============================================================================
# CENTRAL COLUMN: PLOT + ORDER SLIDER
# ==============================================================================

plot_column = VBox(
    [
        fig.canvas,
        order_slider_box
    ],
    layout=Layout(
        width='820px',
        align_items='center',
        justify_content='flex-start'
    )
)

# ==============================================================================
# MAIN LAYOUT
#
# The total width is increased slightly to accommodate the 18 px gap between
# the plot and the Displayed Quantity panel.
# ==============================================================================

main_layout = HBox(
    [
        filter_panel,
        plot_column,
        response_panel
    ],
    layout=Layout(
        width='1320px',
        align_items='flex-start',
        justify_content='flex-start'
    )
)

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)